___
<img style="float: right; margin: 15px 15px 15px 15px;" src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRU6ycfMWHLjnsOPma9wwzKXckhE5iYOGp-dA&s" width="380px" height="200px" />


# <font color= #bbc28d> **Halloween Wikipedia Views** </font>
#### <font color= #2E9AFE> `Examen 3 - MNLP`</font>
- <Strong> Diana Valdivia, Sofía Maldonado, Samantha Sánchez & Rafael Takata. </Strong>
- <Strong> Fecha </Strong>: 13/05/2026.

___

<p style="text-align:right;"> Image retrieved from: https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRU6ycfMWHLjnsOPma9wwzKXckhE5iYOGp-dA&s</p>

# **<font color= #bbc28d> Modelado | Halloween </font>**

## **<font color= #bbc28d> &ensp; • Contexto </font>**

El objetivo de este proyecto es predecir el número de visitas diarias al artículo de *Halloween* en Wikipedia en inglés utilizando modelos de deep learning para series de tiempo.

El principal desafío de la serie es su comportamiento extremadamente estacional. Durante la mayor parte del año, las visitas permanecen uniformes, pero conforme se acerca el 31 de octubre ocurre un incremento que termina en un spike muy pronunciado. Este tipo de comportamiento es difícil de modelar con métodos tradicionales, ya que el evento más importante ocurre solo una vez al año y concentra gran parte de la variabilidad de la serie.

A partir del análisis exploratorio y las features construidas previamente, en este notebook se entrenan y comparan dos arquitecturas de redes neuronales:

- **FFNN (Feed Forward Neural Network)**  
- **LSTM (Long Short-Term Memory)**  

In [1]:
# Importar librerías
import pandas as pd
import numpy as np
import requests
import joblib
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Seeds para reproducibilidad
tf.random.set_seed(42)
np.random.seed(42)

## **<font color= #bbc28d> &ensp; • Transformación Logarítmica </font>**

Como vimos en las gráficas del EDA, la serie tiene un comportamiento bastante extremo: durante casi todo el año las visitas se mantienen bajas, pero en Halloween aparecen picos enormes que multiplican varias veces el tráfico normal.

El problema de esto es que la red neuronal termina viendo el spike como algo desproporcionadamente grande respecto al resto de la serie. Por ejemplo, un día normal puede tener alrededor de `10k` visitas mientras que el 31 de octubre puede superar las `400k`. Esa diferencia hace que el modelo se enfoque demasiado en reducir el error del pico y pierda precisión en los demás días.

Para reducir este efecto se aplica una transformación logarítmica usando `log1p`. Esta transformación no elimina el spike, pero sí comprime la escala para que las diferencias sean más manejables.

Por ejemplo:
- `log1p(10,000)` ≈ `9.2`
- `log1p(270,000)` ≈ `12.5`

El spike sigue siendo el valor más alto de la serie, pero ya no está tan alejado del resto de observaciones.

Esto ayuda a que:
- el entrenamiento sea más estable,
- el scaler distribuya mejor los datos,
- los errores extremos no dominen completamente el aprendizaje,
- y los `sample_weights` funcionen de forma más efectiva. `[Más adelante se explica]`

El log ayuda a que la red pueda aprender tanto los días normales como el spike de Halloween sin que uno destruya al otro durante el entrenamiento.

In [27]:
# Leer datos previamente guardados
df_train = pd.read_csv('halloween_train.csv', index_col=0, parse_dates=True)
df_val = pd.read_csv('halloween_val.csv', index_col=0, parse_dates=True)
df_test  = pd.read_csv('halloween_test.csv',  index_col=0, parse_dates=True)

# Features que seleccionamos
FEATURES = ['views', 'es_halloween', 'semana_halloween', 'dias_para_halloween', 'views_lag7']

df_train = df_train[FEATURES]
df_val = df_val[FEATURES]
df_test  = df_test[FEATURES]

for df in [df_train, df_val, df_test]:
    df['views']      = np.log1p(df['views'].values)
    df['views_lag7'] = np.log1p(df['views_lag7'].values)

# Verificar fechas y cantidades
print(f'Train: {df_train.index.min().date()} - {df_train.index.max().date()} ({len(df_train)} dias)')
print(f'Val: {df_val.index.min().date()} - {df_val.index.max().date()} ({len(df_val)} dias)')
print(f'Test:  {df_test.index.min().date()}  - {df_test.index.max().date()}  ({len(df_test)} dias)')

Train: 2020-01-15 - 2024-09-30 (1721 dias)
Val: 2024-10-01 - 2024-10-31 (31 dias)
Test:  2025-10-01  - 2025-10-31  (31 dias)


## **<font color= #bbc28d> &ensp; • Escalamiento de Datos </font>**
Se aplica `MinMaxScaler` principalmente para mejorar/facilitar el entrenamiento de las redes neuronales. Al escalar todas las variables al rango `[0, 1]`, se evita que variables con escalas grandes dominen y que los gradientes estén mas regulados.

Además, el scaler se ajusta solo en el conjunto de entrenamiento y después se aplica a validación y test usando, evitando el data leakage.

Aunque `views` ya está incluida en el primer scaler, se crea otro scaler separado porque después solo queremos desescalar las predicciones de `views`.

El scaler principal fue entrenado con todas las columnas, así que para usar `inverse_transform()` necesitaría recibir nuevamente todas las variables completas. Como el modelo solo predice `views`, regresarla a su valor original implicaría reconstruir el df en cada prediccion.

Con un scaler especificamente de las views podemos convertir las predicciones de forma más simple.

In [28]:
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(df_train[FEATURES])
val_scaled   = scaler.transform(df_val[FEATURES])
test_scaled  = scaler.transform(df_test[FEATURES])

# Scaler independiente para views (para invertir predicciones)
scaler_views = MinMaxScaler()
scaler_views.fit(df_train[['views']])

print('Escalado OK')

Escalado OK


## **<font color= #bbc28d> &ensp; • Ventanas Deslizantes </font>**
Tanto la FFNN como la LSTM trabajan usando ventanas deslizantes: el modelo recibe varios días consecutivos como entrada para predecir el siguiente día.

En este caso se utilizan ventanas de:
- `45` días para la FFNN
- `30` días para la LSTM

La idea es darle suficiente contexto al modelo para que pueda detectar cómo aumentan las visitas conforme se acerca Halloween.

El tamaño de las ventanas es un `hiperparámetro`, en nuestro caso fueron elegidas de forma arbitraria y a base de muchas pruebas y errores, buscando el tamaño que mejor se ajustara a los datos.

Además, se utilizan las `colas` (`cola_val` y `cola_test`). Estas simplemente agregan los últimos días del conjunto anterior al inicio de validación y test. Esto porque el modelo necesita una ventana completa para hacer la primera predicción. Sin las colas, se perderían los primeros registros de validación y test al no tener suficientes días previos como contexto.

In [ ]:
W_ffnn = 45  # Tamaño para FFNN
W_lstm = 30  # Tamaño para LSTM

def make_windows(data, W):
    X, y = [], []
    for i in range(W, len(data)):
        X.append(data[i-W:i])
        y.append(data[i, 0])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

# Ventanas para FFNN
cola_val_ffnn  = train_scaled[-W_ffnn:]
cola_test_ffnn = val_scaled[-W_ffnn:]
X_train_ffnn, y_train_ffnn = make_windows(train_scaled, W_ffnn)
X_val_ffnn,   y_val_ffnn   = make_windows(np.concatenate([cola_val_ffnn,  val_scaled]),  W_ffnn)
X_test_ffnn,  y_test_ffnn  = make_windows(np.concatenate([cola_test_ffnn, test_scaled]), W_ffnn)

# Ventanas para LSTM
cola_val_lstm  = train_scaled[-W_lstm:]
cola_test_lstm = val_scaled[-W_lstm:]
X_train_lstm, y_train_lstm = make_windows(train_scaled, W_lstm)
X_val_lstm,   y_val_lstm   = make_windows(np.concatenate([cola_val_lstm,  val_scaled]),  W_lstm)
X_test_lstm,  y_test_lstm  = make_windows(np.concatenate([cola_test_lstm, test_scaled]), W_lstm)

In [81]:
N_FEATURES = X_train_ffnn.shape[2]

## <font color= #bbc28d> &ensp; • **Modelado — FFNN** </font>

Una red **Feedforward Neural Network** (FFNN) procesa la entrada en una sola dirección, sin memoria temporal explícita. Recibe la ventana de $W$ días aplanada como un vector y aplica transformaciones no lineales en capas sucesivas:

* **Capa de entrada**: aplana la ventana $(W \times F)$ en un vector de $W \cdot F$ features
* **Capas ocultas**: aplican transformaciones afines seguidas de una activación no lineal
* **Capa de salida**: proyecta a un escalar (la predicción del día siguiente)

La transformación en cada capa $l$ es:

$$
z^{(l)} = W^{(l)} \cdot a^{(l-1)} + b^{(l)}
$$
$$
a^{(l)} = \text{ReLU}(z^{(l)}) = \max(0,\, z^{(l)})
$$

Donde $a^{(0)} = \text{Flatten}(X_t) \in \mathbb{R}^{W \cdot F}$ es la ventana de entrada aplanada.

El **Dropout** aplicado entre capas desactiva aleatoriamente una fracción $p$ de neuronas durante el entrenamiento, actuando como regularizador:

$$
\tilde{a}^{(l)} = a^{(l)} \odot \text{Bernoulli}(1-p) \cdot \frac{1}{1-p}
$$

In [ ]:
def build_ffnn(window, n_features):
    model = keras.Sequential([
        layers.Input(shape=(window, n_features)),
        layers.Flatten(),
        layers.Dense(256, activation='relu'), 
        layers.Dropout(0.1),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.1),
        layers.Dense(64, activation='relu'),
        layers.Dense(1)
    ], name='FFNN')
    return model

## <font color= #bbc28d> &ensp; • **Modelado — LSTM** </font>

Una red **Long Short-Term Memory** (LSTM) extiende las RNN tradicionales con un mecanismo de compuertas que controla qué información conservar o descartar a lo largo del tiempo:

* **Compuerta de olvido** ($f_t$): decide qué olvidar del estado anterior
* **Compuerta de entrada** ($i_t$): decide qué nueva información almacenar
* **Compuerta de salida** ($o_t$): decide qué parte del estado exponer

Las ecuaciones de celda son:

$$
f_t = \sigma(W_f \cdot [h_{t-1},\, x_t] + b_f)
$$
$$
\tilde{C}_t = \tanh(W_C \cdot [h_{t-1},\, x_t] + b_C)
$$
$$
C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t
$$
$$
h_t = o_t \odot \tanh(C_t)
$$


In [ ]:
def build_lstm(n_features, hidden=48):
    model = keras.Sequential([
        layers.Input(shape=(None, n_features)),
        layers.LSTM(hidden),
        layers.Dense(32, activation='relu'),
        layers.Dense(1)
    ])
    return model

In [ ]:
# Construir modelos
ffnn = build_ffnn(W_ffnn, N_FEATURES)
lstm = build_lstm(N_FEATURES)

print(f'FFNN parametros: {ffnn.count_params():,}')
print(f'LSTM parametros: {lstm.count_params():,}')
ffnn.summary()
lstm.summary()

FFNN parametros: 99,073
LSTM parametros: 11,969


Model: "FFNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_36 (Flatten)            │ (None, 225)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_204 (Dense)               │ (None, 256)            │        57,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_78 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_205 (Dense)               │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_79 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_206 (Dense)               │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_207 (Dense)               │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 99,073 (387.00 KB)

 Trainable params: 99,073 (387.00 KB)

...

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_62 (LSTM)                  │ (None, 48)             │        10,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_208 (Dense)               │ (None, 32)             │         1,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_209 (Dense)               │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,969 (46.75 KB)

 Trainable params: 11,969 (46.75 KB)

 Non-trainable params: 0 (0.00 B)

## <font color= #bbc28d> &ensp; • **Entrenamiento y Pesos Ponderados** </font>
Uno de los principales retos de esta serie es que el spike de Halloween representa solo unos pocos días dentro de varios años de datos. Si todas las muestras tuvieran el mismo peso, el modelo aprendería principalmente a predecir los días normales y podría ignorar precisamente el evento más importante.

Para evitar esto, se utilizan `sample_weights`, asignando mayor importancia a los días cercanos a Halloween:

- El 31 de octubre recibe el peso más alto.
- Los últimos días de octubre reciben un peso intermedio.
- El resto del año mantiene peso 1.

De esta manera, el modelo es penalizado más fuertemente cuando falla en el spike o en la rampa previa al evento.

Los pesos utilizados no son exactamente iguales entre la FFNN y la LSTM porque ambos modelos aprenden de forma distinta. La LSTM ya captura parte del contexto temporal gracias a su memoria interna, mientras que la FFNN depende más de las features explícitas y necesita una ponderación más agresiva.

In [ ]:
def make_sample_weights(dates_tr, spike_weight=15.0, ramp_weight=3.0):
    weights = np.ones(len(dates_tr))
    for i, d in enumerate(dates_tr):
        # Cuando es 31 peso spike
        if d.month == 10 and d.day == 31:
            weights[i] = spike_weight
        
        # Cuando se va haciendo el dobles, peso rampa
        elif d.month == 10 and d.day >= 26:
            weights[i] = ramp_weight
    return weights

Además, el entrenamiento utiliza varios callbacks para mejorar estabilidad y evitar sobreajuste:

- `EarlyStopping`: detiene el entrenamiento cuando el `val_loss` deja de mejorar y restaura los mejores pesos.
- `ReduceLROnPlateau`: reduce automáticamente el learning rate cuando el modelo se estanca.
- `LambdaCallback`: imprime métricas durante entrenamiento para monitorear el progreso.

## <font color= #bbc28d> &ensp; • **Función de pérdida — Huber** </font>

La pérdida de **Huber** combina lo mejor de MSE y MAE según la magnitud del error:

$$
L_\delta(y, \hat{y}) = \begin{cases} \dfrac{1}{2}(y - \hat{y})^2 & \text{si } |y - \hat{y}| \leq \delta \\[10pt] \delta \cdot |y - \hat{y}| - \dfrac{1}{2}\delta^2 & \text{si } |y - \hat{y}| > \delta \end{cases}
$$

* Para errores **pequeños** ($|e| \leq \delta$): se comporta como **MSE**, siendo suave y diferenciable cerca del mínimo
* Para errores **grandes** ($|e| > \delta$): se comporta como **MAE**, creciendo linealmente y sin castigar desproporcionadamente los outliers


In [ ]:
# Funcion para entrenar
def train_model(model, X_tr, y_tr, X_val, y_val, dates_tr, epochs=300, lr=1e-3, patience=30, batch_size=32, spike_weight=15, ramp_weight=3): 

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss=keras.losses.Huber(delta=1.0) # Usamos Huber por los picos
    )

    # Quedarse con los mejores pesos
    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=patience,
        restore_best_weights=True,
        verbose=1
    )

    # Reduce lr cuando val_loss se estanca
    reduce_lr = keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.7,
        patience=15,   
        min_lr=1e-5,   # que no baje tanto
        verbose=1
    )

    log_cb = keras.callbacks.LambdaCallback(
        on_epoch_end=lambda epoch, logs: print(
            f'  Epoca {epoch+1:3d} | train: {logs["loss"]:.6f} | val: {logs["val_loss"]:.6f} | lr: {logs.get("learning_rate", lr):.2e}'
        ) if (epoch + 1) % 5 == 0 else None
    )

    # Crear la matriz de pesos
    sample_weights = make_sample_weights(dates_tr, spike_weight=spike_weight, ramp_weight=ramp_weight)

    # Entrenar
    history = model.fit(
        X_tr, y_tr,
        sample_weight=sample_weights,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stop, reduce_lr, log_cb],
        verbose=0
    )

    return history.history['loss'], history.history['val_loss']

# fechas que corresponden a X_train
dates_tr_ffnn = df_train.index[W_ffnn:]
dates_tr_lstm = df_train.index[W_lstm:] 

print('=== Entrenando FFNN ===')
ffnn = build_ffnn(W_ffnn, N_FEATURES)
h_tr_ffnn, h_val_ffnn = train_model(ffnn, X_train_ffnn, y_train_ffnn, X_val_ffnn, y_val_ffnn, dates_tr_ffnn,lr=1e-3,batch_size=32)

print('\n=== Entrenando LSTM ===')
lstm = build_lstm(N_FEATURES)
h_tr_lstm, h_val_lstm = train_model(lstm, X_train_lstm, y_train_lstm, X_val_lstm, y_val_lstm, dates_tr_lstm, spike_weight=10, ramp_weight=4, lr=3e-4, batch_size=16) 


In [ ]:
# Curvas de aprendizaje
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['FFNN — Curva de perdida', 'LSTM — Curva de perdida'])

for col, (h_tr, h_val) in enumerate(
    [(h_tr_ffnn, h_val_ffnn), (h_tr_lstm, h_val_lstm)], start=1
):
    fig.add_trace(go.Scatter(y=h_tr,  name='Train',
                             line=dict(color='#5B6547'), showlegend=(col==1)), row=1, col=col)
    fig.add_trace(go.Scatter(y=h_val, name='Validacion',
                             line=dict(color="#385C6E", dash='dash'), showlegend=(col==1)), row=1, col=col)

fig.update_layout(template='plotly_white', height=350,
                  title='Curvas de perdida')
fig.update_xaxes(title_text='Epoca')
fig.update_yaxes(title_text='Huber')
fig.write_html('Assets/curvas_perdida2.html',include_plotlyjs='cdn', full_html=False)
fig.show()

## <font color= #bbc28d> &ensp; • **Forecast Recursivo** </font>

Para generar predicciones futuras se utiliza una estrategia de forecast recursivo. En lugar de predecir todo el horizonte de una sola vez, el modelo predice un día y esa predicción se reutiliza inmediatamente como parte de la entrada para predecir el siguiente.

Las variables de calendario (`es_halloween`, `semana_halloween`, `dias_para_halloween`) sí se conocen de antemano porque dependen únicamente de la fecha, por lo que pueden utilizarse directamente durante el forecast.

El proceso funciona así:
1. El modelo recibe una ventana inicial de días reales.
2. Predice el siguiente día.
3. Esa predicción se agrega a la ventana.
4. Se elimina el día más antiguo.
5. El proceso se repite hasta completar todo el horizonte.

Sin embargo, tiene una limitación importante: los errores pueden acumularse con el tiempo porque las nuevas predicciones dependen de predicciones anteriores del propio modelo.

In [ ]:
def recursive_forecast(model, seed_window, exog_future_scaled, steps):

    window = seed_window.copy()
    preds  = []

    for i in range(steps):
        # Keras espera (batch, timesteps, features)
        x = np.expand_dims(window, axis=0)          
        pred_scaled = model.predict(x, verbose=0)[0, 0]
        preds.append(pred_scaled)

        new_row = np.array([pred_scaled] + list(exog_future_scaled[i]), dtype=np.float32)
        window  = np.vstack([window[1:], new_row])

    return np.array(preds)


# Columas exógenas
exog_test = test_scaled[:, 1:]  
steps     = len(df_test)

seed_ffnn = train_scaled[-W_ffnn:]
seed_lstm = train_scaled[-W_lstm:]

# Predecir
preds_ffnn_sc = recursive_forecast(ffnn, seed_ffnn, exog_test, steps)
preds_lstm_sc = recursive_forecast(lstm, seed_lstm, exog_test, steps)

# Des-escalar
preds_ffnn = np.expm1(scaler_views.inverse_transform(preds_ffnn_sc.reshape(-1,1)).flatten())
preds_lstm = np.expm1(scaler_views.inverse_transform(preds_lstm_sc.reshape(-1,1)).flatten())

real = np.expm1(scaler_views.inverse_transform(test_scaled[:, 0].reshape(-1, 1)).flatten())

print('Forecast generado OK')

Forecast generado OK


Antes de evaluar los modelos, se revisa el rango de valores predichos por cada arquitectura. Esto permite analizar si las predicciones tienen escalas razonables y si los modelos lograron capturar parcialmente el spike de Halloween.

Comparar mínimos y máximos también ayuda a detectar problemas comunes como:
- Predicciones demasiado suaves.
- Sobreestimación extrema del spike.
- Valores fuera del rango esperado.
- Modelos que colapsan hacia una media constante.

Revisar únicamente métricas agregadas puede ocultar errores importantes en el comportamiento del pico principal.

In [ ]:
# Ver rango de predicciones
print('real:  ', real.min(), real.max())
print('ffnn:  ', preds_ffnn.min(), preds_ffnn.max())
print('lstm:  ', preds_lstm.min(), preds_lstm.max())

real:   8736.0 269292.9999999999
ffnn:   14536.184 238367.42
lstm:   11756.928 261108.56


Los rangos muestran que ambos modelos lograron capturar correctamente la existencia de un spike grande en Halloween, aunque con comportamientos distintos.

- La **FFNN** produce predicciones más suavizadas: sobreestima los días normales (`14k` vs `8.7k`) y subestima el pico máximo (`238k` vs `269k`). Esto sugiere que el modelo tiende a acercarse a una media más estable y le cuesta reproducir cambios tan extremos.

- La **LSTM**, en cambio, logra aproximarse mucho más al máximo real (`261k` vs `269k`) y mantiene un rango más cercano al comportamiento verdadero de la serie. Esto indica que la arquitectura recurrente captura mejor la dinámica temporal y la aceleración previa al spike.

In [ ]:
# Visualizar fechas
fechas = df_test.index

fig = go.Figure()
fig.add_trace(go.Scatter(x=fechas, y=real,
                         name='Real', line=dict(color='black', width=2)))
fig.add_trace(go.Scatter(x=fechas, y=preds_ffnn,
                         name='FFNN', line=dict(color="#95a565", dash='dash', width=2)))
fig.add_trace(go.Scatter(x=fechas, y=preds_lstm,
                         name='LSTM', line=dict(color='#2a9d8f', dash='dot', width=2)))

fig.add_vline(
    x=pd.Timestamp('2025-10-31').timestamp() * 1000,
    line=dict(color='black', dash='dash', width=1),
    annotation_text='Halloween 2025'
)

fig.update_layout(
    title='Forecast — Octubre 2025 | FFNN vs LSTM',
    xaxis_title='Fecha', yaxis_title='Views',
    template='plotly_white',
    legend=dict(x=0.01, y=0.99)
)
fig.write_html('Assets/forecast_recursivo_2.html',include_plotlyjs='cdn', full_html=False)
fig.show()

## <font color= #bbc28d> &ensp; • **Métricas de Performance** </font>
Se utilizan tres métricas para evaluar distintos aspectos del desempeño de los modelos:

| Métrica | ¿Qué mide? | Sensible a... | Interpretación |
|---|---|---|---|
| **MAE** | Diferencia promedio absoluta entre predicción y valor real | Magnitud general del error | Todos los errores tienen un impacto similar |
| **RMSE** | Error promedio dando más peso a errores grandes | Spikes y predicciones extremas | Penaliza fuertemente errores grandes |
| **MAPE** | Error promedio en porcentaje respecto al valor real | Valores reales pequeños | Mide qué tan lejos está la predicción en términos porcentuales |

El uso conjunto de estas métricas permite tener una evaluación más completa. Por ejemplo, un modelo puede tener buen MAE en días normales pero un RMSE alto si falla fuertemente en Halloween.

In [206]:
def compute_metrics(real, preds, nombre):
    mae  = mean_absolute_error(real, preds)
    rmse = np.sqrt(mean_squared_error(real, preds))
    mape = np.mean(np.abs((real - preds) / (real + 1))) * 100
    print(f'{nombre:6s} -> MAE: {mae:>10,.0f} | RMSE: {rmse:>10,.0f} | MAPE: {mape:.1f}%')
    return {'Modelo': nombre, 'MAE': mae, 'RMSE': rmse, 'MAPE (%)': mape}

m_ffnn = compute_metrics(real, preds_ffnn, 'FFNN')
m_lstm = compute_metrics(real, preds_lstm, 'LSTM')

metrics_df = pd.DataFrame([m_ffnn, m_lstm]).set_index('Modelo').round(2)
metrics_df

FFNN   -> MAE:      7,375 | RMSE:     10,111 | MAPE: 34.0%
LSTM   -> MAE:      3,378 | RMSE:      6,826 | MAPE: 12.3%


,MAE,RMSE,MAPE (%)
Modelo,,,
FFNN,7375.39,10111.30,34.02
LSTM,3377.65,6825.92,12.32


Los resultados muestran una diferencia clara entre los dos modelos.

La **FFNN** logra seguir parcialmente el comportamiento general de la serie, pero presenta errores mas grandes, especialmente en el spike de Halloween. Su `MAPE` de `34%` indica que las predicciones suelen alejarse bastante de los valores reales.

Por otro lado, la LSTM obtiene mejores resultados en todas las métricas:
- Reduce el `MAE` de ~7.3k a ~3.3k views.
- Disminuye significativamente el `RMSE`, lo que sugiere un mejor manejo de errores grandes y del spike principal.
- Obtiene un `MAPE` mucho menor (`12.3%`)

En general, estos resultados sugieren que la LSTM logra capturar mejor la dinámica temporal de la serie y la aceleración de Halloween.

In [208]:
fig = make_subplots(rows=1, cols=3, subplot_titles=['MAE', 'RMSE', 'MAPE (%)'])
colores = ['#95a565', '#2a9d8f']
modelos = ['FFNN', 'LSTM']

for col, metrica in enumerate(['MAE', 'RMSE', 'MAPE (%)'], start=1):
    vals = [m_ffnn[metrica], m_lstm[metrica]]
    fig.add_trace(
        go.Bar(x=modelos, y=vals, marker_color=colores,
               text=[f'{v:,.1f}' for v in vals],
               textposition='outside', showlegend=False),
        row=1, col=col
    )

fig.update_layout(title='Comparacion de metricas — FFNN vs LSTM',
                  template='plotly_white', height=500)
fig.write_html('Assets/metricas_comparativa_2.html',include_plotlyjs='cdn', full_html=False)
fig.show()

## <font color= #bbc28d> &ensp; • **Análisis del error por día: spike vs. días normales** </font>
Además de las métricas globales, también se analiza cómo se distribuye el error a lo largo de octubre. Esto es importante porque un modelo podría tener buenas métricas promedio simplemente por acertar en los días normales, pero fallar justamente en Halloween.

Por esa razón, se separa explícitamente el error del 31 de octubre del error promedio en el resto del mes.

Este análisis permite verificar si los `sample_weights` realmente ayudaron al modelo a prestar más atención al spike y a la rampa previa, en lugar de enfocarse únicamente en minimizar el error general de la serie.

La gráfica muestra el error absoluto diario de ambos modelos y permite identificar en qué momentos cada arquitectura falla más.

In [ ]:
errores_ffnn = np.abs(real - preds_ffnn)
errores_lstm = np.abs(real - preds_lstm)

fig = go.Figure()
fig.add_trace(go.Bar(x=fechas, y=errores_ffnn, name='Error FFNN', marker_color='#95a565', opacity=0.8))
fig.add_trace(go.Bar(x=fechas, y=errores_lstm, name='Error LSTM', marker_color='#2a9d8f', opacity=0.8))

fig.add_vline(
    x=pd.Timestamp('2025-10-31').timestamp() * 1000,
    line=dict(color='black', dash='dash'),
    annotation_text='Halloween'
)

fig.update_layout(
    title='Error absoluto por dia — donde falla cada modelo?',
    xaxis_title='Fecha', yaxis_title='Error absoluto (views)',
    barmode='group', template='plotly_white'
)
fig.write_html('Assets/errores_acumulados_2.html',include_plotlyjs='cdn', full_html=False)
fig.show()

spike_idx  = fechas.day == 31
normal_idx = ~spike_idx

print('Error en el spike (31-oct 2025):')
print(f'  FFNN: {errores_ffnn[spike_idx][0]:,.0f} views')
print(f'  LSTM: {errores_lstm[spike_idx][0]:,.0f} views')

print('\nError promedio dias normales (1-30 oct):')
print(f'  FFNN: {errores_ffnn[normal_idx].mean():,.0f} views')
print(f'  LSTM: {errores_lstm[normal_idx].mean():,.0f} views')

Error en el spike (31-oct 2025):
  FFNN: 35,037 views
  LSTM: 8,184 views

Error promedio dias normales (1-30 oct):
  FFNN: 6,453 views
  LSTM: 3,217 views


La gráfica confirma las métricas, la LSTM captura mejor el patrón de los datos sobre la FFNN.

En el día más importante de la serie la FFNN presenta un error de más de `35k` views, mientras que la LSTM reduce ese error a aproximadamente `8k`. Esto indica que la arquitectura recurrente logra capturar mucho mejor la aceleración previa al spike y su magnitud final.

La diferencia también se mantiene en los días normales del mes:
- FFNN: ~`6.4k` views de error promedio.
- LSTM: ~`3.2k` views de error promedio.

Esto sugiere que la LSTM no solo mejora en el spike principal, sino que también modela mejor el comportamiento general de la serie durante octubre. Sin embargo porbaremos ambos modelos ya que una cosa es que sean buenos prediciendo con data que si existe, y otra hacia el futuro.

In [ ]:
# Guardar nuestros modelos con version de corrida
import joblib

# joblib.dump(scaler,        'Models/Scaler/scaler_4.pkl')
# joblib.dump(scaler_views,  'Models/Scaler/scaler_views_4.pkl')

# ffnn.save('Models/ffnn_halloween_4.keras')
# lstm.save('Models/lstm_halloween_4.keras')

print('Scalers guardados OK')
print('Modelos guardados OK')

Scalers guardados OK
Modelos guardados OK


## <font color= #bbc28d> **Predicción a Futuro | Halloween** </font>
Se consulta directamente la **API de Wikimedia** para obtener las vistas reales más recientes (enero–mayo 2026) además de cargar tanto el escalador como los modelos que se entrenaron previamente:

In [220]:
ffnn         = keras.models.load_model('Models/ffnn_halloween_4.keras')
lstm         = keras.models.load_model('Models/lstm_halloween_4.keras')
scaler       = joblib.load('Models/Scaler/scaler_4.pkl')
scaler_views = joblib.load('Models/Scaler/scaler_views_4.pkl')

FEATURES = ['views', 'es_halloween', 'semana_halloween', 'dias_para_halloween', 'views_lag7']
W_ffnn = 45
W_lstm = 30

url = (
    "https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/"
    "en.wikipedia/all-access/user/Halloween/daily/20260101/20260512"
)
r = requests.get(url, headers={"User-Agent": "halloween-forecast/1.0"})
items = r.json()["items"]

df_fresh = pd.DataFrame(items)
df_fresh["date"] = pd.to_datetime(df_fresh["timestamp"], format="%Y%m%d%H")
df_fresh = df_fresh.sort_values("date").set_index("date")[["views"]]
df_fresh

,views
date,
2026-01-01,1797
2026-01-02,1379
2026-01-03,4218
2026-01-04,5029
2026-01-05,1603
...,...
2026-05-07,1252
2026-05-08,1151
2026-05-09,1061


## <font color= #bbc28d> &ensp; • **Construir variables Exógenas** </font>
Para las fechas futuras se construyen las variables exógenas de la misma forma que en el entrenamiento. El lag de 7 días se inicializa con los últimos valores reales disponibles y luego se alimenta con las propias predicciones. 

In [ ]:
def days_to_halloween(date):
    hw = pd.Timestamp(date.year, 10, 31)
    diff = (hw - date).days
    return diff if diff >= 0 else (pd.Timestamp(date.year + 1, 10, 31) - date).days

df_fresh['es_halloween']        = ((df_fresh.index.month == 10) & (df_fresh.index.day == 31)).astype(int)
df_fresh['semana_halloween']    = ((df_fresh.index.month == 10) & (df_fresh.index.day >= 24)).astype(int)
df_fresh['dias_para_halloween'] = df_fresh.index.map(days_to_halloween)
df_fresh['views_lag7']          = df_fresh['views'].shift(7)
df_fresh = df_fresh.dropna()[FEATURES]

# Logaritmos
df_fresh['views']      = np.log1p(df_fresh['views'].values)
df_fresh['views_lag7'] = np.log1p(df_fresh['views_lag7'].values)

print(f"Datos frescos: {df_fresh.index.min().date()} → {df_fresh.index.max().date()} ({len(df_fresh)} días)")

Datos frescos: 2026-01-08 → 2026-05-11 (124 días)


In [ ]:
# Escalar los datos
fresh_scaled = scaler.transform(df_fresh)

# Añadir la cola a los datos
seed_ffnn = fresh_scaled[-W_ffnn:]
seed_lstm = fresh_scaled[-W_lstm:]

In [ ]:
# Construir df de fechas a futuro (como 500 dias maso)
fecha_inicio = pd.Timestamp('2026-05-13')
fecha_fin    = pd.Timestamp('2027-10-31')
fechas_pred  = pd.date_range(fecha_inicio, fecha_fin, freq='D')

# Usar los últimos 7 días reales para los primeros lags
views_reales = df_fresh['views'].values

exog_rows = []
for i, d in enumerate(fechas_pred):
    # valores reales históricos
    if i < 7:
        lag7 = float(views_reales[-(7 - i)])
    else:
        lag7 = 0.0  # placeholder, se sobreescribirá en el forecast

    exog_rows.append({
        'views':               0.0,
        'es_halloween':        int(d.month == 10 and d.day == 31),
        'semana_halloween':    int(d.month == 10 and d.day >= 24),
        'dias_para_halloween': float(days_to_halloween(d)),
        'views_lag7':          lag7
    })

df_exog = pd.DataFrame(exog_rows, index=fechas_pred)
df_exog

,views,es_halloween,semana_halloween,dias_para_halloween,views_lag7
2026-05-13,0.0,0,0,171.0,7.120444
2026-05-14,0.0,0,0,170.0,7.143618
2026-05-15,0.0,0,0,169.0,7.133296
2026-05-16,0.0,0,0,168.0,7.049255
2026-05-17,0.0,0,0,167.0,6.967909
...,...,...,...,...,...
2027-10-27,0.0,0,1,4.0,0.000000
2027-10-28,0.0,0,1,3.0,0.000000
2027-10-29,0.0,0,1,2.0,0.000000
2027-10-30,0.0,0,1,1.0,0.000000


## <font color= #bbc28d> &ensp; • **Predecir el Futuro** </font>
Como en un escenario real no conocemos las views futuras, las predicciones se generan de forma recursiva: el modelo predice un día y luego usa esa misma predicción para avanzar al siguiente. Las variables del calendario sí se conocen de antemano, por lo que ayudan a mantener el contexto temporal conforme avanza octubre y se acerca Halloween. Además de evaluar la FFNN y la LSTM por separado, también se crea un ensamble simple promediando ambas predicciones para intentar obtener resultados más estables.

In [ ]:
def recursive_forecast_exog(model, seed_window, df_exog_raw, scaler, scaler_views, steps):
    window = seed_window.copy()
    preds  = []

    for i in range(steps):
        x       = np.expand_dims(window, axis=0)
        pred_sc = np.clip(model.predict(x, verbose=0)[0, 0], 0, 1) 
        preds.append(pred_sc)
        
        row_raw = df_exog_raw.iloc[i].astype(float).copy()
        
        # inverse_transform -> log -> valor normal
        views_log = scaler_views.inverse_transform([[pred_sc]])[0, 0] 
        row_raw['views'] = views_log 

        if i >= 7:
            lag_log = scaler_views.inverse_transform([[preds[i-7]]])[0, 0] 
            row_raw['views_lag7'] = lag_log 

        row_df     = pd.DataFrame([row_raw], columns=FEATURES)
        row_scaled = scaler.transform(row_df)[0]
        window     = np.vstack([window[1:], row_scaled])

    return np.array(preds)


steps         = len(fechas_pred)
preds_ffnn_sc = recursive_forecast_exog(ffnn, seed_ffnn, df_exog, scaler, scaler_views, steps)
preds_lstm_sc = recursive_forecast_exog(lstm, seed_lstm, df_exog, scaler, scaler_views, steps)

preds_ffnn = np.expm1(scaler_views.inverse_transform(preds_ffnn_sc.reshape(-1,1)).flatten())
preds_lstm = np.expm1(scaler_views.inverse_transform(preds_lstm_sc.reshape(-1,1)).flatten())

preds_ens  = (preds_ffnn + preds_lstm) / 2

In [228]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_fresh.index, y=np.expm1(df_fresh['views'].values),
    name='Real (histórico)', line=dict(color='black', width=1.5)))

fig.add_trace(go.Scatter(
    x=fechas_pred, y=preds_ffnn,
    name='FFNN', line=dict(color='#95a565', width=2)))

fig.add_trace(go.Scatter(
    x=fechas_pred, y=preds_lstm,
    name='LSTM', line=dict(color='#2a9d8f', dash='dash', width=2)))

fig.add_trace(go.Scatter(
    x=fechas_pred, y=preds_ens,
    name='Ensamble', line=dict(color="#956ea4", dash='dot', width=2)))

fig.add_vline(
    x=pd.Timestamp('2026-10-31').timestamp() * 1000,
    line=dict(color='black', dash='dash', width=1),
    annotation_text='Halloween 2026'
)
fig.add_vline(
    x=pd.Timestamp('2027-10-31').timestamp() * 1000,
    line=dict(color='black', dash='dash', width=1),
    annotation_text='Halloween 2027'
)
fig.add_vline(
    x=pd.Timestamp('2026-05-12').timestamp() * 1000,
    line=dict(color='gray', dash='dot', width=1),
    annotation_text='Hoy'
)

fig.update_layout(
    title='Forecast Halloween — Mayo a Octubre 2027',
    xaxis_title='Fecha', yaxis_title='Views',
    template='plotly_white',
    legend=dict(x=0.01, y=0.99)
)
fig.write_html('Assets/predicciones_halloween_2.html',include_plotlyjs='cdn', full_html=False)
fig.show()

In [227]:
print(f"\nPredicción Halloween 2026 (31-oct):")
idx_31 = fechas_pred.get_loc(pd.Timestamp('2026-10-31'))
print(f"  FFNN:     {preds_ffnn[idx_31]:>10,.1f} views")
print(f"  LSTM:     {preds_lstm[idx_31]:>10,.1f} views")
print(f"  Ensamble: {preds_ens[idx_31]:>10,.1f} views")
print(f"\nPredicción Halloween 2027 (31-oct):")
idx_31 = fechas_pred.get_loc(pd.Timestamp('2027-10-31'))
print(f"  FFNN:     {preds_ffnn[idx_31]:>10,.1f} views")
print(f"  LSTM:     {preds_lstm[idx_31]:>10,.1f} views")
print(f"  Ensamble: {preds_ens[idx_31]:>10,.1f} views")


Predicción Halloween 2026 (31-oct):
  FFNN:      274,584.5 views
  LSTM:      262,069.8 views
  Ensamble:  268,327.1 views

Predicción Halloween 2027 (31-oct):
  FFNN:      273,326.2 views
  LSTM:      271,304.2 views
  Ensamble:  272,315.2 views


## <font color= #bbc28d> &ensp; • **Análisis de las Predicciones** </font>

Las predicciones para Halloween 2026 y 2027 hacen sentido con el comportamiento histórico de la serie. Ambos modelos logran mantenerse en el rango típico de los spikes reales de Halloween, alrededor de `260k–275k` views, sin explotar exageradamente ni colapsar hacia valores demasiado bajos.

Algo interesante es que los modelos también parecen captar la ligera tendencia a la baja que tiene la serie en los últimos años. Aunque el spike sigue siendo enorme, las views máximas recientes ya no alcanzan los niveles más altos observados anteriormente, y las predicciones futuras reflejan justamente ese comportamiento más estable y ligeramente descendente.

La FFNN tiende a producir predicciones un poco más altas y suaves, mientras que la LSTM se comporta de manera más conservadora y cercana a lo que vimos en el test de 2025. El ensamble termina funcionando bastante bien porque toma un punto medio entre ambas, reduciendo un poco los errores individuales de cada arquitectura. Aunque la verdad el modelo de LSTM sigue siendo bastante mejor.

## <font color= #bbc28d> &ensp; • **Conclusiones** </font>

Algo que quedó muy claro durante el proyecto es que el problema realmente no era solo “entrenar una red neuronal”, sino entender primero cómo se comportaba la serie. Desde el EDA se veía que Halloween rompía completamente el patrón normal de views y que la mayor parte del año la serie era bastante estable.

También quedó claro que entrenar directamente con las views originales era complicado porque el spike hacía que todo explotara. Por eso la transformación logarítmica terminó siendo tan importante: ayudó a que el modelo pudiera aprender el comportamiento general sin obsesionarse únicamente con el 31 de octubre.

Las variables de calendario también hicieron mucha diferencia. Features simples como `dias_para_halloween` o `semana_halloween` terminaron aportando mucho más que variables típicas como el día de la semana.
